
# XGBoost Predictive Maintenance (Fair Comparison Framework)


## Step 1 — Data Loading and Preparation

In [71]:
import sys
sys.path.append(".")

In [72]:
import os
print(os.getcwd())

/Users/martinwalterampah/Desktop/Final_Research_Project


In [73]:
import importlib
import Scripts.RUSBoostTrain

from Scripts.XGBoostTrain import train_xgboost

%load_ext autoreload
%autoreload 2
importlib.reload(Scripts.XGBoostTrain)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


<module 'Scripts.XGBoostTrain' from '/Users/martinwalterampah/Desktop/Final_Research_Project/Scripts/XGBoostTrain.py'>

In [74]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler

%load_ext autoreload
%autoreload 2

CSV_PATH = "Data_file/ai4i2020.csv"

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)
df.head()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Dataset shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [75]:
#model, X_test, y_test, y_pred_xgb, y_proba_xgb, results, cfg, feature_map = train_xgboost("configs/XGBoost/XGtrain.yaml")
#model, X_test, y_test, y_pred_xgb, y_proba_xgb, results, cfg, feature_name_mapping = train_xgboost("configs/XGBoost/XGtrain.yaml")
xgb_raw, xgb_calibrated, X_test, y_test, y_pred_xgb, y_proba_xgb, results, cfg, mapping = train_xgboost("configs/XGBoost/XGtrain.yaml")


Chosen Threshold: 0.01
Calibration Recall: 0.9853


In [76]:
print("Chosen threshold:", results["threshold"])
print("PR-AUC:", results["pr_auc"])
print("ROC-AUC:", results["roc_auc"])

Chosen threshold: 0.01
PR-AUC: 0.6951967547023826
ROC-AUC: 0.9668584825234442


In [77]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, digits=4))

print("\nROC AUC:", roc_auc_score(y_test, y_proba_xgb))
print("Average Precision (PR AUC):", average_precision_score(y_test, y_proba_xgb))

print("\nChosen Threshold:", results["threshold"])

Confusion Matrix:
[[1675  257]
 [   5   63]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9970    0.8670    0.9275      1932
           1     0.1969    0.9265    0.3247        68

    accuracy                         0.8690      2000
   macro avg     0.5969    0.8967    0.6261      2000
weighted avg     0.9698    0.8690    0.9070      2000


ROC AUC: 0.9668584825234442
Average Precision (PR AUC): 0.6951967547023826

Chosen Threshold: 0.01


In [78]:
# =========================================================
# XGBoost Experiment Tracking (timestamped run)
# =========================================================

import os
import json
import yaml
from datetime import datetime
import joblib
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, average_precision_score

# --- 1. Timestamp + run dir ---
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
run_dir = os.path.join("experiments_xgboost", f"run_{timestamp}")
os.makedirs(run_dir, exist_ok=True)

print(f"[XGBoost] Run dir: {run_dir}")

# --- 2. Save model (optional) ---
model_path = os.path.join(run_dir, "xgboost_calibrated.pkl")
#joblib.dump(model, model_path)
joblib.dump(xgb_raw, os.path.join(run_dir, "xgb_raw.pkl"))
joblib.dump(xgb_calibrated, os.path.join(run_dir, "xgb_calibrated.pkl"))

print(f"Model saved: {model_path}")

# --- 3. Save config ---
config_src = "configs/XGBoost/XGtrain.yaml"
config_dst = os.path.join(run_dir, "XGtrain.yaml")

with open(config_src, "r") as f:
    cfg_data = yaml.safe_load(f)

with open(config_dst, "w") as f:
    yaml.dump(cfg_data, f)

print(f"Config saved: {config_dst}")

# --- 4. Save metrics ---
metrics = {
    "model": "XGBoost",
    "confusion_matrix": confusion_matrix(y_test, y_pred_xgb).tolist(),
    "classification_report": classification_report(y_test, y_pred_xgb, output_dict=True),
    "roc_auc": roc_auc_score(y_test, y_proba_xgb),
    "pr_auc": average_precision_score(y_test, y_proba_xgb),
    "threshold": results["threshold"]
}

metrics_path = os.path.join(run_dir, "metrics.json")

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=4)

print(f"Metrics saved: {metrics_path}")

# --- 5. Create notes file ---
notes_path = os.path.join(run_dir, "notes.txt")

with open(notes_path, "w") as f:
    f.write(
        "XGBoost Experiment Notes\n"
        "========================\n\n"
        f"Timestamp: {timestamp}\n\n"
        "Observations:\n- \n\n"
        "Changes Made:\n- \n\n"
        "Next Steps:\n- \n"
    )

print(f"Notes file created: {notes_path}")

[XGBoost] Run dir: experiments_xgboost/run_2026-04-21_18-02-54
Model saved: experiments_xgboost/run_2026-04-21_18-02-54/xgboost_calibrated.pkl
Config saved: experiments_xgboost/run_2026-04-21_18-02-54/XGtrain.yaml
Metrics saved: experiments_xgboost/run_2026-04-21_18-02-54/metrics.json
Notes file created: experiments_xgboost/run_2026-04-21_18-02-54/notes.txt


In [13]:

selected_uids = {
    "TWF": [2672, 3866, 6341, 8358, 9019],
    "HDF": [3237, 4079, 4174, 4327, 4502],
    "PWF": [464, 1493, 3001, 7537, 8583],
    "OSF": [250, 3020, 5400, 7592, 9660],
}

all_selected_uids = [uid for group in selected_uids.values() for uid in group]

selected_cases = df[df["UDI"].isin(all_selected_uids)].copy().sort_values("UDI")

selected_cases[["UDI", "Type", "Machine failure", "TWF", "HDF", "PWF", "OSF"]]


,UDI,Type,Machine failure,TWF,HDF,PWF,OSF
249,250,L,1,0,0,0,1
463,464,L,1,0,0,1,0
1492,1493,M,1,0,0,1,0
2671,2672,M,1,1,0,0,0
3000,3001,H,1,0,0,1,0
3019,3020,L,1,0,0,0,1
3236,3237,M,1,0,1,0,0
3865,3866,H,1,1,0,0,0
4078,4079,H,1,0,1,0,0
4173,4174,M,1,0,1,0,0
